# Phase 7 — Results Tables for Paper
Generates all publication-ready tables.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import wilcoxon
from collections import Counter
print('Imports successful')

Imports successful


In [3]:
BASE    = Path('E:/project1/EW_MOGA_Project')
RESULTS = BASE / 'results'
print('Configuration set')

FEATURE_COLS = ['Mean','Variance','Standard Deviation','Entropy',
    'Skewness','Kurtosis','Contrast','Energy',
    'ASM','Homogeneity','Dissimilarity','Correlation','Coarseness']

Configuration set


In [4]:
df_all = pd.read_csv(RESULTS / 'all_results_merged.csv')
print(f'Loaded: {df_all.shape}')
print(f'Methods: {df_all.method.unique().tolist()}')

Loaded: (105, 10)
Methods: ['EW_MOGA', 'Standard_GA', 'NSGA2_Parsimony', 'SHAP_RF', 'Boruta', 'RFE', 'mRMR']


In [5]:
# ── Table 2 — Classification accuracy ────────────────────────
order = ['Standard_GA','NSGA2_Parsimony','SHAP_RF','EW_MOGA',
         'Boruta','RFE','mRMR']
order = [m for m in order if m in df_all.method.unique()]

t2 = df_all.groupby('method').agg(
    Accuracy =('accuracy',   lambda x: f'{x.mean():.4f} ± {x.std():.4f}'),
    Features =('n_features', lambda x: f'{x.mean():.2f} ± {x.std():.2f}'),
    Runtime  =('runtime_sec',lambda x: f'{x.mean():.1f}'),
).reindex(order)
print('Table 2 — Classification Accuracy:')
print(t2.to_string())
t2.to_csv(RESULTS / 'table2_accuracy.csv')
print('Saved table2_accuracy.csv')

Table 2 — Classification Accuracy:
                        Accuracy      Features Runtime
method                                                
Standard_GA      0.9929 ± 0.0031   7.47 ± 1.64   474.8
NSGA2_Parsimony  0.9907 ± 0.0035   4.87 ± 1.41   296.2
SHAP_RF          0.8351 ± 0.0123   6.00 ± 0.00     4.8
EW_MOGA          0.9902 ± 0.0035   5.73 ± 1.33   742.8
Boruta           0.9872 ± 0.0038  12.00 ± 0.00    77.6
RFE              0.9845 ± 0.0025   6.00 ± 0.00     9.2
mRMR             0.9829 ± 0.0040   6.00 ± 0.00    10.1
Saved table2_accuracy.csv


In [6]:
# ── Table 3 — Audit metrics ──────────────────────────────────
t3 = df_all.groupby('method').agg(
    CCC=('audit_ccc', lambda x: f'{x.mean():.4f} ± {x.std():.4f}'),
    PR =('audit_pr',  lambda x: f'{x.mean():.4f} ± {x.std():.4f}'),
).reindex(order)
print('Table 3 — Post-hoc Audit Results:')
print(t3.to_string())
t3.to_csv(RESULTS / 'table3_audit.csv')
print('Saved table3_audit.csv')

Table 3 — Post-hoc Audit Results:
                             CCC               PR
method                                           
Standard_GA      0.0733 ± 0.2745  0.8638 ± 0.0753
NSGA2_Parsimony  0.3382 ± 0.3272  0.9305 ± 0.1011
SHAP_RF          0.0660 ± 0.2008  0.1383 ± 0.3462
EW_MOGA          0.6210 ± 0.1451  0.9651 ± 0.0324
Boruta           0.4120 ± 0.0521  0.9438 ± 0.0203
RFE              0.5092 ± 0.0939  0.9208 ± 0.0222
mRMR             0.5790 ± 0.1223  0.9776 ± 0.0154
Saved table3_audit.csv


In [7]:
# ── Table 4 — Wilcoxon tests ─────────────────────────────────
df_w = pd.read_csv(RESULTS / 'wilcoxon_results.csv')
print('Table 4 — Wilcoxon Signed-Rank Results:')
print(df_w.to_string(index=False))
print('(Already saved as wilcoxon_results.csv)')

Table 4 — Wilcoxon Signed-Rank Results:
   metric        baseline  W_statistic  p_value  n_pairs  significant
 accuracy     Standard_GA         29.5 0.083161       15        False
 accuracy NSGA2_Parsimony         50.0 0.875200       15        False
 accuracy         SHAP_RF          0.0 0.000653       15         True
 accuracy          Boruta          0.0 0.002174       15         True
 accuracy             RFE          0.0 0.000648       15         True
 accuracy            mRMR          0.0 0.000647       15         True
audit_ccc     Standard_GA          1.0 0.000122       15         True
audit_ccc NSGA2_Parsimony         22.0 0.030151       15         True
audit_ccc         SHAP_RF          1.0 0.000122       15         True
audit_ccc          Boruta          2.0 0.000183       15         True
audit_ccc             RFE         20.0 0.023068       15         True
audit_ccc            mRMR         48.0 0.495434       15        False
 audit_pr     Standard_GA          1.0 0.000122   

In [8]:
# ── Table 5 — Feature frequency ──────────────────────────────
ew_rows = df_all[df_all['method']=='EW_MOGA']
all_feats = []
for _, row in ew_rows.iterrows():
    if pd.notna(row['feature_names']) and row['feature_names']:
        all_feats.extend([f.strip() for f in str(row['feature_names']).split(',') if f.strip()])
feat_counts = Counter(all_feats)
total = len(ew_rows)
t5 = pd.DataFrame([
    {'Feature': f, 'Count': feat_counts.get(f,0),
     'Frequency (%)': f'{100*feat_counts.get(f,0)/total:.1f}%'}
    for f in FEATURE_COLS
]).sort_values('Count', ascending=False)
print('Table 5 — Feature Selection Frequency (EW-MOGA):')
print(t5.to_string(index=False))
t5.to_csv(RESULTS / 'table5_feature_frequency.csv', index=False)
print('Saved table5_feature_frequency.csv')

Table 5 — Feature Selection Frequency (EW-MOGA):
           Feature  Count Frequency (%)
          Kurtosis     14         93.3%
       Correlation     12         80.0%
           Entropy      9         60.0%
          Skewness      8         53.3%
              Mean      7         46.7%
        Coarseness      6         40.0%
               ASM      6         40.0%
          Variance      5         33.3%
Standard Deviation      5         33.3%
          Contrast      5         33.3%
            Energy      4         26.7%
       Homogeneity      3         20.0%
     Dissimilarity      2         13.3%
Saved table5_feature_frequency.csv


In [9]:
# ── Table 6 — Ablation study ─────────────────────────────────
abl_path = RESULTS / 'ablation' / 'ablation_results.csv'
if abl_path.exists():
    df_abl = pd.read_csv(abl_path)
    t6 = df_abl.groupby('variant').agg(
        Accuracy=('accuracy',  lambda x: f'{x.mean():.4f} ± {x.std():.4f}'),
        CCC     =('audit_ccc', lambda x: f'{x.mean():.4f} ± {x.std():.4f}'),
        PR      =('audit_pr',  lambda x: f'{x.mean():.4f} ± {x.std():.4f}'),
        Features=('n_features',lambda x: f'{x.mean():.2f}'),
    )
    print('Table 6 — Ablation Study:')
    print(t6.to_string())
    t6.to_csv(RESULTS / 'table6_ablation.csv')
    print('Saved table6_ablation.csv')
else:
    print('Ablation results not found — run Notebook 4 first')

Table 6 — Ablation Study:
                         Accuracy              CCC               PR Features
variant                                                                     
FID_LFS           0.9903 ± 0.0034  0.6256 ± 0.0985  0.9375 ± 0.0636     7.00
FID_only          0.9901 ± 0.0035  0.6057 ± 0.1126  0.9639 ± 0.0392     7.20
Full_SCI_FID_LFS  0.9903 ± 0.0028  0.7220 ± 0.1301  0.9679 ± 0.0314     6.33
LFS_only          0.9901 ± 0.0030  0.6088 ± 0.1189  0.9532 ± 0.0426     7.07
SCI_FID           0.9904 ± 0.0030  0.7013 ± 0.1887  0.9670 ± 0.0315     6.00
SCI_LFS           0.9904 ± 0.0031  0.6119 ± 0.1882  0.9711 ± 0.0316     5.00
SCI_only          0.9902 ± 0.0027  0.6382 ± 0.2041  0.9733 ± 0.0301     5.47
Saved table6_ablation.csv


In [10]:
print('='*55)
print('All tables generated and saved to results/')
print('='*55)
print()
print('Notebook 7 complete — ready to run Notebook 8')

All tables generated and saved to results/

Notebook 7 complete — ready to run Notebook 8


In [1]:
import pandas as pd
from pathlib import Path

RESULTS = Path('E:/project1/EW_MOGA_Project/results')

df = pd.read_csv(RESULTS / 'all_results_merged.csv')

methods = ['Boruta', 'RFE', 'mRMR', 'EW_MOGA']
methods = [m for m in methods if m in df.method.unique()]

print('New Baselines Summary:')
print('='*65)
for method in methods:
    sub = df[df.method==method]
    print(f'{method}:')
    print(f'  Accuracy  : {sub.accuracy.mean():.4f} ± {sub.accuracy.std():.4f}')
    print(f'  CCC       : {sub.audit_ccc.mean():.4f} ± {sub.audit_ccc.std():.4f}')
    print(f'  PR        : {sub.audit_pr.mean():.4f} ± {sub.audit_pr.std():.4f}')
    print(f'  N features: {sub.n_features.mean():.2f} ± {sub.n_features.std():.2f}')
    print(f'  Runtime   : {sub.runtime_sec.mean():.1f}s')
    print()

New Baselines Summary:
Boruta:
  Accuracy  : 0.9872 ± 0.0038
  CCC       : 0.4120 ± 0.0521
  PR        : 0.9438 ± 0.0203
  N features: 12.00 ± 0.00
  Runtime   : 77.6s

RFE:
  Accuracy  : 0.9845 ± 0.0025
  CCC       : 0.5092 ± 0.0939
  PR        : 0.9208 ± 0.0222
  N features: 6.00 ± 0.00
  Runtime   : 9.2s

mRMR:
  Accuracy  : 0.9829 ± 0.0040
  CCC       : 0.5790 ± 0.1223
  PR        : 0.9776 ± 0.0154
  N features: 6.00 ± 0.00
  Runtime   : 10.1s

EW_MOGA:
  Accuracy  : 0.9902 ± 0.0035
  CCC       : 0.6210 ± 0.1451
  PR        : 0.9651 ± 0.0324
  N features: 5.73 ± 1.33
  Runtime   : 742.8s



In [2]:
import pandas as pd
from pathlib import Path

RESULTS = Path('E:/project1/EW_MOGA_Project/results')
df = pd.read_csv(RESULTS / 'all_results_merged.csv')

print('All methods in merged file:', df.method.unique().tolist())
print()

for m in ['Boruta', 'RFE', 'mRMR']:
    if m in df.method.unique():
        sub = df[df.method==m]
        print(f'{m}:')
        print(f'  Accuracy  : {sub.accuracy.mean():.4f} ± {sub.accuracy.std():.4f}')
        print(f'  CCC       : {sub.audit_ccc.mean():.4f} ± {sub.audit_ccc.std():.4f}')
        print(f'  PR        : {sub.audit_pr.mean():.4f} ± {sub.audit_pr.std():.4f}')
        print(f'  N features: {sub.n_features.mean():.2f} ± {sub.n_features.std():.2f}')
        print()
    else:
        print(f'{m}: NOT found in merged file')
        print('  Run Notebook 05 first or check new_baselines_results.csv')
        print()

All methods in merged file: ['EW_MOGA', 'Standard_GA', 'NSGA2_Parsimony', 'SHAP_RF', 'Boruta', 'RFE', 'mRMR']

Boruta:
  Accuracy  : 0.9872 ± 0.0038
  CCC       : 0.4120 ± 0.0521
  PR        : 0.9438 ± 0.0203
  N features: 12.00 ± 0.00

RFE:
  Accuracy  : 0.9845 ± 0.0025
  CCC       : 0.5092 ± 0.0939
  PR        : 0.9208 ± 0.0222
  N features: 6.00 ± 0.00

mRMR:
  Accuracy  : 0.9829 ± 0.0040
  CCC       : 0.5790 ± 0.1223
  PR        : 0.9776 ± 0.0154
  N features: 6.00 ± 0.00



In [3]:
# Check if new baselines file exists separately
nb5_path = RESULTS / 'new_baselines_results.csv'
if nb5_path.exists():
    df_nb5 = pd.read_csv(nb5_path)
    print('new_baselines_results.csv found:')
    print(df_nb5.method.value_counts())
    for m in df_nb5.method.unique():
        sub = df_nb5[df_nb5.method==m]
        print(f'{m}: CCC={sub.audit_ccc.mean():.4f}±{sub.audit_ccc.std():.4f} PR={sub.audit_pr.mean():.4f}±{sub.audit_pr.std():.4f}')
else:
    print('new_baselines_results.csv not found - run Notebook 05 first')

new_baselines_results.csv found:
method
Boruta    15
RFE       15
mRMR      15
Name: count, dtype: int64
Boruta: CCC=0.4120±0.0521 PR=0.9438±0.0203
RFE: CCC=0.5092±0.0939 PR=0.9208±0.0222
mRMR: CCC=0.5790±0.1223 PR=0.9776±0.0154
